# Filter keys

### 1. Remove rotamers with missing chi angles

Check which rotamers have missing chi angles

In [ ]:
import json
import os
import pandas as pd
from rotadel.common.config import ROOT_DIR
from rotadel.common.constants import NUMBER_OF_CHI_ANGLES

data_folder = ROOT_DIR / "data"
os.environ["DATA_FOLDER"] = str(data_folder)  # for the %%bash cell below

# Dunbrack library
with open(f"{data_folder}/angles_Dunbrack.json", "r") as f:
    dunbrack_data = json.load(f)

# Print rotamers with less defined chi angles than expected
ids_missing_chi = []
for rot_id, props in dunbrack_data.items():
    letter = props["letter"]
    expected = NUMBER_OF_CHI_ANGLES.get(letter, 0)
    chi_values = [props.get(f"chi{i+1}") for i in range(4)]
    actual = sum(x is not None for x in chi_values)
    if actual < expected:
        ids_missing_chi.append(rot_id)
        print(f"{rot_id}: {props['prob']}")

# Save as CSV
pd.DataFrame(ids_missing_chi, columns=["key"]).to_csv(
    f"{data_folder}/keys_missing_chi.csv", index=False, header=False
)

Remove these rotamers from the keys of the whole library

In [ ]:
df_initial = pd.read_csv(f"{data_folder}/keys_species_initial.csv", names=["key"])
missing_chi_df = pd.read_csv(f"{data_folder}/keys_missing_chi.csv", names=["key"])

# Strip the charge info
df_initial["key_without_species"] = df_initial["key"].str.split("c").str[0]

# Filter out keys with missing chi angles
df_final = df_initial[~df_initial["key_without_species"].isin(missing_chi_df["key"])]
print(f"Removed: {len(df_initial) - len(df_final)} keys")
df_final

Removed: 54 keys


,key,key_without_species
0,Ra-180a-180r1221c+1,Ra-180a-180r1221
1,Ra-180a-180r1222c+1,Ra-180a-180r1222
2,Ra-180a-180r1223c+1,Ra-180a-180r1223
3,Ra-180a-180r1211c+1,Ra-180a-180r1211
4,Ra-180a-180r1212c+1,Ra-180a-180r1212
...,...,...
1077398,Va180a170r3000c0,Va180a170r3000
1077399,Va180a170r2000c0,Va180a170r2000
1077400,Va180a180r1000c0,Va180a180r1000
1077401,Va180a180r3000c0,Va180a180r3000


### 2. Remove rotamer with phi or psi == 180° (same entries as -180°)

In [ ]:
# Print rotamers with phi or psi == 180°
keys_180 = []
rest_keys = []
for rot_id, _ in dunbrack_data.items():
    if "a180" in rot_id:
        keys_180.append(rot_id)
    else:
        rest_keys.append(rot_id)
print(keys_180)
print(
    f"Keys without species info: {len(keys_180)} with a 180° angle, {len(rest_keys)} without\n"
)

# Filter these keys out
df_previous = df_final.copy()
df_final = df_previous[df_previous["key_without_species"].isin(rest_keys)]
print(f"Removed: {len(df_previous) - len(df_final)} keys")
df_final

['Ra-180a180r1221', 'Ra-180a180r1222', 'Ra-180a180r1223', 'Ra-180a180r1211', 'Ra-180a180r1212', 'Ra-180a180r1232', 'Ra-180a180r1213', 'Ra-180a180r1233', 'Ra-180a180r3212', 'Ra-180a180r1322', 'Ra-180a180r1122', 'Ra-180a180r2222', 'Ra-180a180r1323', 'Ra-180a180r1231', 'Ra-180a180r3323', 'Ra-180a180r2212', 'Ra-180a180r2211', 'Ra-180a180r2232', 'Ra-180a180r1321', 'Ra-180a180r1332', 'Ra-180a180r3233', 'Ra-180a180r1333', 'Ra-180a180r3222', 'Ra-180a180r1121', 'Ra-180a180r3232', 'Ra-180a180r2223', 'Ra-180a180r2122', 'Ra-180a180r2233', 'Ra-180a180r2112', 'Ra-180a180r3223', 'Ra-180a180r1123', 'Ra-180a180r3322', 'Ra-180a180r3332', 'Ra-180a180r2221', 'Ra-180a180r3211', 'Ra-180a180r2231', 'Ra-180a180r3221', 'Ra-180a180r2213', 'Ra-180a180r1112', 'Ra-180a180r2111', 'Ra-180a180r3333', 'Ra-180a180r1331', 'Ra-180a180r3321', 'Ra-180a180r1111', 'Ra-180a180r1312', 'Ra-180a180r3312', 'Ra-180a180r1132', 'Ra-180a180r2332', 'Ra-180a180r2121', 'Ra-180a180r1113', 'Ra-180a180r2113', 'Ra-180a180r3231', 'Ra-180a180

,key,key_without_species
0,Ra-180a-180r1221c+1,Ra-180a-180r1221
1,Ra-180a-180r1222c+1,Ra-180a-180r1222
2,Ra-180a-180r1223c+1,Ra-180a-180r1223
3,Ra-180a-180r1211c+1,Ra-180a-180r1211
4,Ra-180a-180r1212c+1,Ra-180a-180r1212
...,...,...
1077284,Va170a160r3000c0,Va170a160r3000
1077285,Va170a160r2000c0,Va170a160r2000
1077286,Va170a170r1000c0,Va170a170r1000
1077287,Va170a170r3000c0,Va170a170r3000


### 3. Remove negative histidines (optimisation issues and very rare)

In [ ]:
df_previous = df_final.copy()
mask_his_neg = df_previous["key"].str.startswith("H") & df_previous["key"].str.endswith(
    "c-1"
)
df_final = df_previous[~mask_his_neg]
print(f"Removed: {len(df_previous) - len(df_final)} keys")
df_final

Removed: 46656 keys


,key,key_without_species
0,Ra-180a-180r1221c+1,Ra-180a-180r1221
1,Ra-180a-180r1222c+1,Ra-180a-180r1222
2,Ra-180a-180r1223c+1,Ra-180a-180r1223
3,Ra-180a-180r1211c+1,Ra-180a-180r1211
4,Ra-180a-180r1212c+1,Ra-180a-180r1212
...,...,...
1077284,Va170a160r3000c0,Va170a160r3000
1077285,Va170a160r2000c0,Va170a160r2000
1077286,Va170a170r1000c0,Va170a170r1000
1077287,Va170a170r3000c0,Va170a170r3000


### 4. Add non-rotameric AAs

In [5]:
# Add alanine and glycine (not in Dunbrack library because no chi angles)
single_aas = pd.DataFrame(
    [
        {"key": "Aa0a0r0000c0", "key_without_species": "Aa0a0r0000"},
        {"key": "Ga0a0r0000c0", "key_without_species": "Ga0a0r0000"},
    ]
)
df_previous = df_final.copy()
df_final = pd.concat([df_previous, single_aas], ignore_index=True)
print(f"Added: {len(df_final) - len(df_previous)} keys")
df_final

Added: 2 keys


,key,key_without_species
0,Ra-180a-180r1221c+1,Ra-180a-180r1221
1,Ra-180a-180r1222c+1,Ra-180a-180r1222
2,Ra-180a-180r1223c+1,Ra-180a-180r1223
3,Ra-180a-180r1211c+1,Ra-180a-180r1211
4,Ra-180a-180r1212c+1,Ra-180a-180r1212
...,...,...
973242,Va170a170r1000c0,Va170a170r1000
973243,Va170a170r3000c0,Va170a170r3000
973244,Va170a170r2000c0,Va170a170r2000
973245,Aa0a0r0000c0,Aa0a0r0000


### 5. Save as CSV

In [6]:
print(f"Number of rotamers kept: {len(df_final)}")
df_final.to_csv(
    f"{data_folder}/keys_species_cleaned.csv",
    columns=["key"],
    index=False,
    header=False,
)

Number of rotamers kept: 973247


# Prepare batches

In [ ]:
%%bash
# Split the dataset in 150 batches of 6490 rotamers each
points_per_batch=6490
whole_csv="${DATA_FOLDER}/keys_species_cleaned.csv"
batches_folder="${DATA_FOLDER}/batches_keys_species_6490_cleaned"
mkdir -p ${batches_folder}
split -l ${points_per_batch} -d -a 3 ${whole_csv} ${batches_folder}/batch_